# Predicción de Rendimiento de Estudiantes

Proyecto de regresión listo para **Google Colab**.

**Pregunta:** ¿podemos predecir la nota final de un estudiante a partir de factores demográficos y de estudio?

**Target:** `G3` (nota final, 0-20)

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset clásico Student Performance (UCI).

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
url = "https://raw.githubusercontent.com/arunk13/MSDA-Assignments/master/IS607Fall2015/Assignment3/student-mat.csv"
df = pd.read_csv(url, sep=";")
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df["G3"].describe())
df.info()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["G3"], kde=True, bins=20)
plt.title("Distribución de nota final (G3)")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.boxplot(data=df, x="studytime", y="G3", ax=axes[0])
axes[0].set_title("Tiempo de estudio vs G3")
sns.boxplot(data=df, x="failures", y="G3", ax=axes[1])
axes[1].set_title("Reprobaciones vs G3")
sns.scatterplot(data=df, x="G1", y="G3", ax=axes[2])
axes[2].set_title("G1 vs G3")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="higher", hue=df["G3"] > 10)
plt.title("Quiere educación superior vs aprueba")
plt.show()

### Hallazgos
- Las notas previas (G1, G2) correlacionan muy fuerte con G3.
- Más reprobaciones → peores notas finales.
- Tiempo de estudio tiene efecto moderado.

## 3. Preparar datos

In [ ]:
X = df.drop(columns=["G3"])
y = df["G3"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos

In [ ]:
modelos = {
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados.append({
        "modelo": nombre,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred, squared=False),
        "R2": r2_score(y_test, y_pred),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("R2", ascending=False)

## 5. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)

plt.figure(figsize=(8, 5))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Top features — Random Forest")
plt.tight_layout()
plt.show()

## 6. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_scaled = rf_pipe.named_steps["scaler"].transform(X_train)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, feature_names=X.columns, max_display=10, show=False)
plt.title("SHAP Summary — Estudiantes")
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, feature_names=X.columns, plot_type="bar", max_display=10, show=False)
plt.title("Importancia media SHAP")
plt.tight_layout()
plt.show()

## 7. Conclusiones

- Las notas previas (G1 y G2) dominan la predicción de G3.
- R² suele estar en **0.80–0.90** en test.
- Variables de comportamiento (failures, studytime) tienen efecto secundario.

**Rango esperado:** R² ~0.82–0.88 | MAE ~1.5–2.0